# Dynamic Pricing Simulation for Ride-Hailing Platforms

This notebook demonstrates an end-to-end pipeline for:
1. **Data Generation** — Realistic synthetic ride-hailing data
2. **Demand Prediction** — Linear Regression, Random Forest, XGBoost
3. **Dynamic Pricing** — Surge multiplier based on demand/supply
4. **Simulation** — Revenue, rides served, summary statistics
5. **Visualization** — Demand vs supply, surge over time, price distribution

## 1. Setup and Imports

In [ ]:
import sys
from pathlib import Path

# Add project root to path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import matplotlib.pyplot as plt
from src.data_processing import (
    generate_synthetic_data,
    aggregate_to_hourly,
    prepare_ml_features,
    load_and_preprocess,
    train_test_split_temporal,
)
from src.demand_model import evaluate_models_with_split, get_best_model
from src.pricing_model import apply_pricing, PricingConfig
from src.visualization import generate_all_plots

## 2. Load and Prepare Data

In [ ]:
# Load data (tries NYC taxi download, falls back to synthetic)
base = Path.cwd().parent
df, data_source = load_and_preprocess(
    use_nyc_download=True,
    data_dir=base / "data",
    n_days_synthetic=7,
)
print(f"Data source: {data_source}")
print(f"Data shape: {df.shape}")
df.head(10)

## 3. Train Demand Prediction Models

In [ ]:
# Train/test split (time-based)
df_train, df_test = train_test_split_temporal(df, test_ratio=0.2)
X_train, y_train = prepare_ml_features(df_train)
X_test, y_test = prepare_ml_features(df_test)

models, train_metrics, test_metrics = evaluate_models_with_split(
    X_train, y_train, X_test, y_test
)
best_model_name = get_best_model(test_metrics, criterion='RMSE')

print("Model Performance (Test Set - Out-of-Sample):")
for name, m in test_metrics.items():
    print(f"  {name}: MAE={m['MAE']:.2f}, RMSE={m['RMSE']:.2f}, R²={m['R2']:.3f}")
print(f"\nBest model: {best_model_name}")

X_full, _ = prepare_ml_features(df)
df['demand_predicted'] = models[best_model_name].predict(X_full)

## 4. Apply Dynamic Pricing

In [ ]:
config = PricingConfig(min_multiplier=1.0, max_multiplier=3.0)
df = apply_pricing(df, config, demand_col='demand_predicted')

df['rides_served'] = df[['demand', 'driver_supply']].min(axis=1)
df['revenue'] = df['rides_served'] * df['price']

df[['timestamp', 'demand', 'driver_supply', 'surge_multiplier', 'price', 'revenue']].head(10)

## 5. Summary Statistics

In [ ]:
total_revenue = df['revenue'].sum()
avg_price = df['price'].mean()
demand_served_ratio = df['rides_served'].sum() / df['demand'].sum()

print(f"Total Revenue:        ${total_revenue:,.2f}")
print(f"Average Price:        ${avg_price:.2f}")
print(f"Demand Served Ratio:  {demand_served_ratio:.2%}")

## 6. Visualizations

In [ ]:
figures_dir = Path.cwd().parent / 'figures'
generate_all_plots(df, figures_dir)
print(f"Figures saved to {figures_dir}")